# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Emmanuel Nkunim Amoah Owusu-Marfo
**Student ID:** 35622028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

/content/lab-4-llm-decision-support
On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.
from google.colab import userdata
API_KEY = userdata.get("API_key")

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content, response.usage

# TODO: Call it once with a simple question and print the answer.
answer, usage = ask_llm("What is the capital of Germany?")
print(answer)
# TODO: Print response.usage as well — how many tokens did your call consume?
print(usage)


The capital of Germany is Berlin.
CompletionUsage(completion_tokens=8, prompt_tokens=48, total_tokens=56, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.049192504, prompt_time=0.001422301, completion_time=0.011194682, total_time=0.012616983)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
1. system role gives the AI instructions about how it should behave. Example: "You are a helpful assistant." The user role contains the specific question the user wants the AI to answer. An example of this is "What is the capital of Germany?"
2. A token is a small piece of text that an LLM processes. API providers bill based on tokens because different requests require very different amounts of computation.


### Part 1.2 — Temperature: the randomness dial

In [3]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

# Temperature = 0.0
print("=== Temperature 0.0 ===")
for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=0.0
    )
    print(f"{i+1}. {answer}")

# Temperature = 1.2
print("\n=== Temperature 1.2 ===")
for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=1.2
    )
    print(f"{i+1}. {answer}")

# TODO: Print all 10 answers, grouped by temperature.

=== Temperature 0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4. **Sika Kokoo**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Kokoo" means "gather" or "collect", so this name could encourage market traders to save and collect their earnings.
5. **Market Booster**: This name suggests that the savings product can help market traders boost their businesses and achieve their financial goals.
6. **Kae Dzi**: "Kae Dzi" is a Ghanaian phrase that means "save for the future". This name could appeal to market traders who are looking to plan for their future and s

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:**

At temperature 0.0, the model produced the same answer almost every time.
At temperature 1.2, the responses from the model are varied which makes the answers less predictable.
For a loan decision-support system, temperature 0.0 is appropriate because loan decisions require consistency and predictability. A higher temperature, in this case 1.2, could introduce unnecessary variation into financial decisions.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_V1 = "Summarize this: {letter_text}"
for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    prompt = SUMMARY_V1.format(letter_text=letter_text)

    response = client.responses.create(model = MODEL, input = prompt)

    print(f"\n----V1 | {letter_id}----")
    print(response.output_text)

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
# SUMMARY_PROMPT_V2

SUMMARY_SYSTEM_V2 = """
You are an assistant to a microfinance loan officer.

Summarize loan applications accurately and neutrally.
Use only information explicitly stated in the application.
Do not invent, assume, or infer facts that are not provided.
Keep the summary to 3-4 sentences.
Include the applicant's name, requested amount, purpose of the loan,
financial information, and repayment/collateral information when available.
"""

SUMMARY_PROMPT_V2 = """
Summarize this loan application:

{letter_text}
"""

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    prompt = SUMMARY_PROMPT_V2.format(letter_text=letter_text)

    response = client.responses.create(
        model=MODEL,
        instructions=SUMMARY_SYSTEM_V2,
        input=prompt,
        temperature=0
    )

    print(f"\n--- V2 | {letter_id} ---")
    print(response.output_text)

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.


----V1 | L002----
Kwame Boateng, a commercial driver in Kumasi, is requesting GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's facing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can. He doesn't have collateral to offer at the moment.

----V1 | L006----
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan in one year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

--- V2 | L002 ---
Kwame Boateng is applying for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. As a commercial driver in Kumasi, he expects his business to improve after the festive season. Kwame does not have collateral to 

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**

1. Since V1 only says "Summarize this", its output can be too general or even omit important information or introduce interpretations that were not stated. For example, the loan letter, L006, actually says he is "22 and full of energy" and that his friends say he is very business minded, but V1 interprets this as " He is confident he can repay". V2 however fixes this by remaining factual and neutral, using information explicitly stated in the application and avoiding inventing details.
2. This is because for a loan decision-support system, incorrect information could influence a financial decision. This failure mode is called hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [6]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0
import json
import pandas as pd

EXTRACT_PROMPT = """
You are an information extraction assistant for a microfinance loan officer.

Extract information from the loan application and return ONLY a valid JSON object.

The JSON object MUST contain EXACTLY these six keys:
{{
    "applicant_name": "string",
    "amount_ghs": number,
    "purpose": "string",
    "monthly_profit_ghs": number or null,
    "has_collateral_or_guarantor": boolean,
    "repayment_months": number or null
}}

Rules:
1. Use only information explicitly stated in the letter.
2. If a field is not stated in the letter, use null.
3. Do not guess, infer, or invent any information.
4. amount_ghs must be a number, not a string.
5. monthly_profit_ghs must be a number if explicitly stated, otherwise null.
6. repayment_months must be a number if explicitly stated, otherwise null.
7. has_collateral_or_guarantor must be true if the applicant explicitly mentions
   collateral or a guarantor, and false if they explicitly state that they have none.
8. Return ONLY the JSON object. Do not include explanations, comments, or markdown.

Worked example:

Letter:
"Dear Loan Officer,
My name is Emmanuel Nkunim. I run a small fast-food joint and am requesting GHS 20,000
to purchase a commercial oven and rent a building. My business earns a monthly profit of GHS 3,500.
My brother will guarantee the loan. I propose to repay the loan over 12 months."

Output:
{{
    "applicant_name": "Emmanuel Nkunim",
    "amount_ghs": 20000,
    "purpose": "purchase a commercial oven and rent a building",
    "monthly_profit_ghs": 3500,
    "has_collateral_or_guarantor": true,
    "repayment_months": 12
}}

Now extract the fields from this loan application:

{letter_text}
"""
# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt,
            temperature=0
        )

        result = response.output_text.strip()

        # Remove Markdown JSON fences if the model returns them
        if result.startswith("```json"):
            result = result[len("```json"):].strip()

        if result.startswith("```"):
            result = result[len("```"):].strip()

        if result.endswith("```"):
            result = result[:-3].strip()

        # Parse the JSON
        extracted = json.loads(result)

        # Check that the required keys are present
        required_keys = {
            "applicant_name",
            "amount_ghs",
            "purpose",
            "monthly_profit_ghs",
            "has_collateral_or_guarantor",
            "repayment_months"
        }

        if set(extracted.keys()) != required_keys:
            print("Warning: JSON does not contain exactly the required keys.")
            return None

        return extracted

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM output as JSON.")
        return None

    except Exception as e:
        print(f"Warning: Extraction failed: {e}")
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)

df = pd.DataFrame(results)

# Put letter_id first
columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

df = df[columns]

display(df)

,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

1. To avoid data leakage. The model should not see the actual test letters as examples because this could bias the extraction and make the evaluation unreliable.

2. Without this instruction, the model may hallucinate or infer missing information instead of leaving it blank. For example, it may create a repayment period or profit value that was never stated.

3. Temperature=0 makes outputs more consistent and deterministic, which is ideal for structured extraction. Creative tasks benefit from higher temperatures because they need more variety and originality.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [7]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.
BRIEF_PROMPT = """
You are an assistant supporting a human microfinance loan officer.

Review the loan application and the extracted information below.

Your task is to produce a concise decision-support brief with exactly these four sections:

1. Strengths
- List important positive factors supported directly by the letter.

2. Risks / Red Flags
- List potential concerns or warning signs supported directly by the letter.
- Do not invent or assume information.

3. Missing Information
- List information or documents the loan officer should request before making a decision.

4. Suggested Next Step
- Suggest an appropriate action such as:
  "invite for interview",
  "request documents",
  "request additional financial records",
  "flag for senior review",
  or another appropriate follow-up action.
- Do NOT recommend "approve" or "reject".

Important:
- Base your analysis only on the information provided.
- Do not invent facts.
- Clearly distinguish stated facts from concerns or missing information.
- Final loan decisions must always be made by a human loan officer.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
def generate_brief(letter_text, extracted_json):
    prompt = BRIEF_PROMPT.format(
        letter_text=letter_text,
        extracted_json=json.dumps(extracted_json, indent=2)
    )

    response = client.responses.create(
        model=MODEL,
        input=prompt,
        temperature=0
    )

    return response.output_text.strip()

briefs = {}

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        briefs[letter_id] = generate_brief(letter_text, extracted)

for letter_id in ["L001", "L002", "L006"]:
    print(f"\n{'='*60}")
    print(f"BRIEF FOR {letter_id}")
    print(f"{'='*60}")
    print(briefs[letter_id])


BRIEF FOR L001
## Step 1: Strengths
The applicant, Akosua Mensah, has a stable business history, selling provisions at Makola Market for 12 years. She has a consistent monthly profit of GHS 900 and has saved GHS 2,500 through the susu scheme over two years without missing a contribution. Additionally, she has a guarantor, her sister, who is a teacher.

## Step 2: Risks / Red Flags
There are no direct red flags mentioned in the application. However, the expansion into frozen foods could introduce new risks, such as increased operational costs or market demand uncertainty, but these are not explicitly stated in the provided information.

## Step 3: Missing Information
The loan officer may need more information about the applicant's business plan, including how she intends to manage the new deep freezer and frozen food inventory, projected increased profits from the expansion, and details about her sister's ability to act as a guarantor. Additionally, financial records or tax returns to 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**

1. Yes. For L003, the system identified strengths such as strong monthly profit, sales records and a fixed deposit. For L006, it flagged the lack of business history, no collateral, no stated profit, and an  overly broad business plan as risks.

2. Practical: The model may make a decision based on incomplete or incorrectly extracted information.

   Ethical: Loan decisions can significantly affect a person's life, so a final decision should be made by a qualified human who can consider real life context.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:**
49b86c65ffee9f1b14444441bd236512ec5793ef

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [13]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

gold_letters = ["L001", "L003", "L006"]

comparison = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in gold_letters:
        predicted = df.loc[
            df["letter_id"] == letter_id, field
        ].iloc[0]

        expected = GOLD[letter_id][field]

        # Case-insensitive comparison for names
        if field == "applicant_name":
            correct = str(predicted).strip().lower() == str(expected).strip().lower()

        # Exact comparison for all other fields
        else:
            correct = predicted == expected

        row[letter_id] = "✓" if correct else "✗"

        if correct:
            correct_count += 1

    row["accuracy"] = correct_count / len(gold_letters)
    comparison.append(row)

comparison_df = pd.DataFrame(comparison)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
comparison_df["accuracy"] = comparison_df["accuracy"].apply(
    lambda x: f"{x:.1%}"
)

display(comparison_df)

display(df[df["letter_id"].isin(["L001", "L003", "L006"])])


,field,L001,L003,L006,accuracy
0,applicant_name,✓,✓,✓,100.0%
1,amount_ghs,✓,✓,✓,100.0%
2,purpose,✗,✗,✗,0.0%
3,monthly_profit_ghs,✓,✓,✗,66.7%
4,has_collateral_or_guarantor,✓,✓,✓,100.0%
5,repayment_months,✓,✓,✓,100.0%


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


### Part 4.2 — Reliability: is the system consistent?

In [14]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.
def extract_fields(letter_text, temperature=0):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt,
            temperature=temperature
        )

        result = response.output_text.strip()

        # Remove Markdown JSON fences
        if result.startswith("```json"):
            result = result[len("```json"):].strip()

        if result.startswith("```"):
            result = result[len("```"):].strip()

        if result.endswith("```"):
            result = result[:-3].strip()

        extracted = json.loads(result)

        required_keys = {
            "applicant_name",
            "amount_ghs",
            "purpose",
            "monthly_profit_ghs",
            "has_collateral_or_guarantor",
            "repayment_months"
        }

        if set(extracted.keys()) != required_keys:
            print("Warning: JSON does not contain exactly the required keys.")
            return None

        return extracted

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM output as JSON.")
        return None

    except Exception as e:
        print(f"Warning: Extraction failed: {e}")
        return None

letter = LETTERS["L004"]

results_by_temp = {}

for temperature in [0, 1.0]:
    runs = []

    for i in range(5):
        result = extract_fields(letter, temperature=temperature)
        runs.append(result)

    results_by_temp[temperature] = runs
# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

for temperature, runs in results_by_temp.items():

    # Valid JSON results
    valid_results = [r for r in runs if r is not None]

    # Convert results to comparable strings
    unique_results = set(
        json.dumps(r, sort_keys=True)
        for r in valid_results
    )

    print(f"\nTemperature = {temperature}")
    print(f"Valid JSON: {len(valid_results)}/5")
    print(f"Unique outputs: {len(unique_results)}")
    print(f"Identical across all valid runs: {len(unique_results) == 1}")


Temperature = 0
Valid JSON: 5/5
Unique outputs: 1
Identical across all valid runs: True

Temperature = 1.0
Valid JSON: 5/5
Unique outputs: 1
Identical across all valid runs: True


### Part 4.3 — Hallucination probing

In [15]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
test1_letter = LETTERS["L001"]

test1_prompt = f"""
You are reviewing this loan application:

{test1_letter}

What is the applicant's credit score?

If the credit score is not stated in the application, clearly say that it is not provided.
Do not guess or invent a credit score.
"""

response1 = client.responses.create(
    model=MODEL,
    input=test1_prompt,
    temperature=0
)

test1_output = response1.output_text.strip()

print("TEST 1 — Missing information")
print("=" * 50)
print(test1_output)
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

irrelevant_text = """
Today's weather forecast: The weather will be partly cloudy with temperatures
around 28 degrees Celsius. There may be light rain in the afternoon.
Winds will be moderate from the southwest. Humidity will remain high.
"""

test2_output = extract_fields(irrelevant_text)

print("TEST 2 — Irrelevant input")
print("=" * 50)
print(json.dumps(test2_output, indent=2))

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
# "TEST 1 -- PASS"
# "The applicant's credit score is not provided."

# "TEST 2 -- PASS"
# "{
#   "applicant_name": null,
#   "amount_ghs": null,
#   "purpose": null,
#   "monthly_profit_ghs": null,
#   "has_collateral_or_guarantor": null,
#   "repayment_months": null
# }"



TEST 1 — Missing information
The applicant's credit score is not provided.
TEST 2 — Irrelevant input
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**

1. The highest accuracy was 100% for applicant name, amount, collateral/guarantor, and repayment months. Monthly profit was 66.7%, while purpose had 0% exact-match accuracy because the model often used different wording even when the meaning was correct. Purpose was therefore the hardest field under exact string matching.

2. Temperature 0 produced more consistent outputs, while temperature 1.0 introduced more variation. This shows that production extraction systems should use low temperature to improve consistency and predictability.

3. No, it did not hallucinate. It clearly stated that the credit score was not provided and returned null for all fields when given the irrelevant weather information.


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**

1. Applicants with poor English writing skills could be unfairly rejected even if they have profitable, well-managed businesses. The system may interpret poor grammar or limited detail as higher risk, creating language and socioeconomic bias.

2. Before deploying in Ghana, I would check the API provider's data-use and retention policies, security measures, where the data is stored and whether appropriate legal safeguards are in place.

3. Human review: this would require a qualified loan officer to review the application's result before any final decision.
   Monitoring and appeal: this would regularly check for bias while also giving applicants the chance to appeal the decision if they believe it was unfair

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.